# 🎮 Deep Convolutional Q-Network (DCQN) — Teaching AI to Play Ms. Pac-Man
---
**What this notebook does:**  
We will train an Artificial Intelligence agent to play the classic arcade game **Ms. Pac-Man** — completely on its own — using a technique called **Deep Q-Learning**.

By the end of this notebook, the AI will learn (by itself, through trial and error) how to:
- Eat dots to earn points
- Avoid ghosts
- Make smart decisions in real time

**No prior deep learning experience needed.** Every single step is explained in plain English. 🚀

---
### 🧠 Key Idea in One Sentence
> We show the AI the game screen (as pixels), and it figures out which button to press to get the most points — just like a human would learn by playing.

---
### 📚 Table of Contents
1. Install Required Packages
2. Import Libraries
3. Understand the Game Environment
4. Build the Neural Network (The AI's "Brain")
5. Preprocess Game Frames (The AI's "Eyes")
6. Build the DCQN Agent (The AI's "Decision Maker")
7. Train the Agent
8. Visualise the Trained Agent Playing


---
## 📦 Section 1 — Install Required Packages

**What we're doing here:**  
Before we can start, we need to install some special software tools. Think of this like installing apps on your phone before you can use them.

Here's what each package does:

| Package | What it does |
|---|---|
| `gymnasium` | Provides the Ms. Pac-Man game environment |
| `ale-py` | The Atari Learning Environment — runs classic Atari games |
| `gymnasium[atari]` | Connects Gymnasium with Atari games |
| `autorom` | Downloads the actual Atari game ROM files (legally) |
| `gymnasium[box2d]` | Extra physics-based environments (good to have) |

> ⏱️ This cell takes **2–4 minutes** to run. Please wait until it finishes before moving on.


In [ ]:
# ── Step 1: Upgrade pip (Python's package manager) to the latest version ──────
!pip install --upgrade pip

# ── Step 2: Remove any old version of gymnasium to avoid conflicts ─────────────
!pip uninstall -y gymnasium

# ── Step 3: Install core Gymnasium (the game framework) ───────────────────────
!pip install gymnasium

# ── Step 4: Install ALE — the engine that actually runs Atari games ────────────
!pip install ale-py

# ── Step 5: Add Atari game support to Gymnasium ───────────────────────────────
!pip install "gymnasium[atari]"

# ── Step 6: Install AutoROM and accept the license to download game files ─────
!pip install autorom[accept-rom-license]
!AutoROM --accept-license

# ── Step 7: Install Box2D physics support (optional but useful) ───────────────
!apt-get install -y swig
!pip install "gymnasium[box2d]"

print("\n✅ All packages installed successfully!")


---
## 📚 Section 2 — Import Libraries

**What we're doing here:**  
Now we "import" all the tools we need into our Python session. This is like opening all the apps you'll use before starting a project.

Here's a plain-English breakdown of each import:

| Import | Plain-English Role |
|---|---|
| `os`, `random` | Basic utilities — file paths, random number generation |
| `numpy` | Fast math with large arrays of numbers |
| `torch` | PyTorch — the deep learning framework we use to build the AI brain |
| `torch.nn` | Neural network building blocks (layers, activations) |
| `torch.optim` | Optimisation algorithms (how the AI learns from mistakes) |
| `torch.nn.functional` | Functions like ReLU, MSE Loss |
| `deque` | A memory buffer that automatically forgets old experiences |
| `PIL / torchvision` | Image processing — resize and convert game frames to tensors |
| `gymnasium / ale_py` | The game environment |


In [ ]:
import os
import random
import numpy as np

# PyTorch — deep learning framework
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# deque = a list with a maximum size; old items are dropped automatically
from collections import deque

# Image processing tools
from PIL import Image
from torchvision import transforms

# The game environment
import gymnasium as gym
import ale_py

print(f"PyTorch version : {torch.__version__}")
print(f"GPU available   : {torch.cuda.is_available()}")
print(f"Device          : {'GPU (CUDA)' if torch.cuda.is_available() else 'CPU'}")


---
## 🕹️ Section 3 — Set Up the Game Environment

**What we're doing here:**  
We create the Ms. Pac-Man game that our AI will play inside. Think of this as launching the game, except the AI sees it instead of you.

**Key concepts:**
- **Observation space** — What the AI *sees* as input. Here it's the raw RGB pixel image of the screen, shaped `(210, 160, 3)` meaning 210 pixels tall × 160 pixels wide × 3 colour channels (Red, Green, Blue).
- **Action space** — What the AI can *do*. These are discrete button presses (e.g. move up, move down, move left, move right, stay still, etc.). With `full_action_space=False`, we reduce this to only the meaningful actions.

> 🎯 **Analogy:** Imagine you can only see a photo of the game screen, and your only controls are a few button options. The AI is in exactly this situation!


In [ ]:
# Register Atari game environments with Gymnasium
gym.register_envs(ale_py)

# Quick check: list a few registered Atari environments to confirm it worked
atari_envs = [e for e in gym.registry.keys() if "ALE" in e]
print(f"Sample Atari environments available: {atari_envs[:5]}")

# ── Create the Ms. Pac-Man game environment ────────────────────────────────────
# full_action_space=False → only expose the useful actions (9 actions instead of 18)
env = gym.make("ALE/MsPacman-v5", full_action_space=False)

# ── Inspect the environment ────────────────────────────────────────────────────
state_shape   = env.observation_space.shape   # (210, 160, 3)
number_action = env.action_space.n             # e.g. 9

print(f"\nGame screen shape  : {state_shape}")
print(f"  → Height         : {state_shape[0]} pixels")
print(f"  → Width          : {state_shape[1]} pixels")
print(f"  → Colour channels: {state_shape[2]} (R, G, B)")
print(f"Number of actions  : {number_action}")
print(f"  (These are the buttons the AI can press)")


---
## ⚙️ Section 4 — Set Hyperparameters

**What we're doing here:**  
Hyperparameters are the "settings" of our training process. They control how fast the AI learns, how much it explores, and how it values future rewards.

Think of it like tuning the settings on a study schedule:
- How many hours do you study per session? → **Learning rate**
- How many past lessons do you review at once? → **Minibatch size**
- How much do you value long-term goals vs. quick wins? → **Discount factor**

| Hyperparameter | Value | What It Means |
|---|---|---|
| `LEARNING_RATE` | 0.0005 | How big a step the AI takes when updating its weights. Too large = unstable. Too small = slow. |
| `MINIBATCH_SIZE` | 64 | How many past experiences the AI reviews each time it learns |
| `DISCOUNT_FACTOR` (γ) | 0.99 | How much the AI values *future* rewards. 0.99 = very far-sighted |
| `MEMORY_SIZE` | 10,000 | Max number of past game moments stored in memory |
| `SOFT_UPDATE_TAU` (τ) | 0.001 | How fast the *target network* (explained later) catches up to the main network |


In [ ]:
# ── Hyperparameters ────────────────────────────────────────────────────────────

LEARNING_RATE   = 5e-4    # 0.0005 — step size for gradient descent
MINIBATCH_SIZE  = 64      # number of experiences sampled per learning step
DISCOUNT_FACTOR = 0.99    # gamma γ: how much future rewards matter (0=none, 1=fully)
MEMORY_SIZE     = 10_000  # replay buffer max size
SOFT_UPDATE_TAU = 1e-3    # tau τ: soft-update blend factor for target network

print("Hyperparameters set:")
print(f"  Learning rate   : {LEARNING_RATE}")
print(f"  Minibatch size  : {MINIBATCH_SIZE}")
print(f"  Discount factor : {DISCOUNT_FACTOR}")
print(f"  Memory size     : {MEMORY_SIZE:,}")
print(f"  Soft-update tau : {SOFT_UPDATE_TAU}")


---
## 👁️ Section 5 — Preprocess Game Frames (The AI's Eyes)

**What we're doing here:**  
The raw game frame is a big `(210, 160, 3)` image. Before feeding it to our neural network we:
1. **Resize** it to `128 × 128` pixels — a fixed, square size our network expects
2. **Convert** it to a PyTorch **tensor** (a multi-dimensional array of numbers that the GPU can work with)
3. **Add a batch dimension** — the network expects a group (batch) of images even if we only have one, so we make it shape `(1, 3, 128, 128)`

> 🧩 **Analogy:** It's like scanning a photo, cropping it to a standard passport size, and converting it to a digital file before a computer can process it.

**Why 128×128?**  
Our neural network's first layer is designed to accept `128×128` images. Smaller is faster to train; we keep enough detail for the AI to see the maze, dots, and ghosts.


In [ ]:
def preprocess_frame(frame):
    """
    Convert a raw RGB game frame (numpy array shape 210×160×3)
    into a normalised PyTorch tensor of shape (1, 3, 128, 128).

    Steps:
      1. Wrap the numpy array in a PIL Image object
      2. Resize to 128×128 (fixed input size for our CNN)
      3. Convert to tensor → pixel values scaled from [0, 255] to [0.0, 1.0]
      4. Add a batch dimension (unsqueeze) → shape becomes (1, 3, 128, 128)
    """
    frame = Image.fromarray(frame)       # numpy array → PIL Image

    transform = transforms.Compose([
        transforms.Resize((128, 128)),   # Resize: 210×160 → 128×128
        transforms.ToTensor()            # PIL Image → tensor, values in [0, 1]
    ])

    return transform(frame).unsqueeze(0)  # Add batch dim: (3,128,128) → (1,3,128,128)


# ── Quick test ──────────────────────────────────────────────────────────────────
sample_state, _ = env.reset()
sample_tensor   = preprocess_frame(sample_state)

print(f"Raw frame shape     : {sample_state.shape}  (height × width × RGB)")
print(f"Processed tensor    : {sample_tensor.shape}  (batch × channels × H × W)")
print(f"Pixel value range   : [{sample_tensor.min():.2f}, {sample_tensor.max():.2f}]")


---
## 🧠 Section 6 — Build the Neural Network (The AI's Brain)

**What we're doing here:**  
We define the neural network that will look at each game frame and decide which action to take.

### Why a *Convolutional* Neural Network (CNN)?
Normal neural networks work well with flat lists of numbers. But game frames are *images* — 2D grids of pixels where nearby pixels are related. CNNs are specially designed to understand images by scanning them with small filters, just like your eyes scan a scene.

### Architecture Overview
```
Input image (128×128 RGB)
       ↓
Conv Layer 1 → 32 filters → BatchNorm → ReLU
       ↓
Conv Layer 2 → 64 filters → BatchNorm → ReLU
       ↓
Conv Layer 3 → 64 filters → BatchNorm → ReLU
       ↓
Conv Layer 4 → 128 filters → BatchNorm → ReLU
       ↓
Flatten (turn 2D map into 1D list)
       ↓
Fully Connected Layer 1 (512 neurons) → ReLU
       ↓
Fully Connected Layer 2 (256 neurons) → ReLU
       ↓
Output Layer (one Q-value per action)
```

### What is a Q-value?
A **Q-value** is a score that answers: *"If I'm in this situation and I take this action, how much total reward can I expect to earn?"*

The AI picks the action with the **highest Q-value** — i.e., the action it thinks will earn the most points.

### What is BatchNorm?
**Batch Normalisation** stabilises training by keeping the numbers flowing through the network from getting too big or too small. Think of it as keeping the volume at a comfortable level on a speaker.


In [ ]:
class Network(nn.Module):
    """
    Deep Convolutional Q-Network (DCQN).

    INPUT : A preprocessed game frame — tensor of shape (batch, 3, 128, 128)
    OUTPUT: Q-values for every possible action — tensor of shape (batch, action_size)

    The network has two stages:
      1. Convolutional layers  → extract visual features from the frame
      2. Fully-connected layers → map those features to action Q-values
    """

    def __init__(self, action_size, seed=42):
        super(Network, self).__init__()
        self.seed = torch.manual_seed(seed)   # For reproducibility

        # ── STAGE 1: Convolutional "eyes" ─────────────────────────────────────
        # Each Conv2d(in_channels, out_channels, kernel_size, stride) scans the
        # image with a small sliding window (kernel) to detect patterns.
        # Stride > 1 shrinks the spatial size of the feature map.

        self.conv1 = nn.Conv2d(3,  32, kernel_size=8, stride=4)  # 3 RGB → 32 feature maps
        self.bn1   = nn.BatchNorm2d(32)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=4, stride=2)  # 32 → 64 features
        self.bn2   = nn.BatchNorm2d(64)

        self.conv3 = nn.Conv2d(64, 64, kernel_size=3, stride=1)  # 64 → 64 features
        self.bn3   = nn.BatchNorm2d(64)

        self.conv4 = nn.Conv2d(64, 128, kernel_size=3, stride=1) # 64 → 128 features
        self.bn4   = nn.BatchNorm2d(128)

        # After 4 conv layers on 128×128 input → output is 10×10×128 = 12,800 values
        # You can verify: 128 → 31 → 14 → 12 → 10 (applying kernel/stride math)

        # ── STAGE 2: Fully-connected decision layers ───────────────────────────
        # These take the flattened visual features and output Q-values.

        self.fc1 = nn.Linear(10 * 10 * 128, 512)  # 12,800 → 512
        self.fc2 = nn.Linear(512, 256)             # 512 → 256
        self.fc3 = nn.Linear(256, action_size)     # 256 → Q-value per action

    def forward(self, state):
        """
        Forward pass: take a game frame, return Q-values for each action.
        ReLU (Rectified Linear Unit) is our activation function — it simply
        sets any negative value to 0, adding non-linearity to the network.
        """
        x = F.relu(self.bn1(self.conv1(state)))   # Conv1 → BN → ReLU
        x = F.relu(self.bn2(self.conv2(x)))        # Conv2 → BN → ReLU
        x = F.relu(self.bn3(self.conv3(x)))        # Conv3 → BN → ReLU
        x = F.relu(self.bn4(self.conv4(x)))        # Conv4 → BN → ReLU

        # Flatten: convert the 3D feature map (10×10×128) into a 1D vector (12800,)
        x = x.view(x.size(0), -1)

        x = F.relu(self.fc1(x))   # Fully connected 1
        x = F.relu(self.fc2(x))   # Fully connected 2

        # Output layer: NO activation — Q-values can be any real number (including negative)
        return self.fc3(x)


# ── Sanity check: verify the network's output shape ───────────────────────────
test_net    = Network(number_action)
test_input  = torch.zeros(1, 3, 128, 128)   # Fake single frame
test_output = test_net(test_input)

print(f"Network input shape  : {test_input.shape}")
print(f"Network output shape : {test_output.shape}  (1 Q-value per action)")
print(f"Number of parameters : {sum(p.numel() for p in test_net.parameters()):,}")


---
## 🤖 Section 7 — Build the DCQN Agent

**What we're doing here:**  
The **Agent** is the full AI system — it combines the neural network with the logic for learning from experience.

### Three Core Ideas

#### 1️⃣ Two Networks: Local & Target
We keep **two copies** of the neural network:
- **Local network** — this is the one that actually gets trained (updated every step)
- **Target network** — a slowly-updated copy used to compute learning targets

**Why two?** Without a stable target, the AI would be "chasing a moving goalposts" — trying to hit a target that changes every step, making learning chaotic. The target network moves *slowly*, giving the local network a stable reference.

#### 2️⃣ Experience Replay (Memory)
Instead of learning from each game moment immediately and forgetting it, we store past experiences in a **memory buffer** (a `deque`). During training we randomly sample a **minibatch** of past experiences to learn from.

**Why random?** Consecutive game frames are heavily correlated (one frame looks almost identical to the next). Learning from them in order would cause the AI to overfit to recent situations. Random sampling breaks this correlation.

#### 3️⃣ Epsilon-Greedy Exploration
The AI faces a classic dilemma:
- **Exploit** — use what it already knows to pick the best action
- **Explore** — try random actions to discover something better

We control this with **epsilon (ε)**:
- Early training: ε is high (e.g. 1.0) → mostly random exploration
- Later training: ε is low (e.g. 0.01) → mostly using learned knowledge
- ε **decays** over time, gradually shifting from exploration to exploitation


In [ ]:
class Agent:
    """
    DCQN Agent — the complete AI system.

    Responsibilities:
      1. Store experiences in memory (experience replay)
      2. Choose actions using epsilon-greedy policy
      3. Learn from sampled batches of experience
      4. Keep the target network slowly updated (soft update)
    """

    def __init__(self, action_size):
        # ── Device: use GPU if available, otherwise CPU ────────────────────────
        self.device      = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        self.action_size = action_size

        print(f"Agent initialised on: {self.device}")

        # ── Two networks ───────────────────────────────────────────────────────
        # local  = the one we train every step
        # target = the stable reference used to compute Q-targets
        self.local_qnetwork  = Network(action_size).to(self.device)
        self.target_qnetwork = Network(action_size).to(self.device)

        # Start both networks with identical weights
        self.target_qnetwork.load_state_dict(self.local_qnetwork.state_dict())

        # ── Optimiser: Adam adjusts weights to minimise loss ───────────────────
        self.optimizer = optim.Adam(self.local_qnetwork.parameters(), lr=LEARNING_RATE)

        # ── Replay memory ──────────────────────────────────────────────────────
        # deque(maxlen=N) automatically drops the oldest item when it gets full
        self.memory = deque(maxlen=MEMORY_SIZE)

    # ──────────────────────────────────────────────────────────────────────────
    def step(self, state, action, reward, next_state, done):
        """
        Called after EVERY game step (every frame).

        1. Preprocess and store the experience in memory
        2. If enough memories exist, trigger a learning update

        Parameters:
          state      — current game frame (numpy array)
          action     — action the agent took (integer)
          reward     — points earned for that action (float)
          next_state — resulting game frame (numpy array)
          done       — True if the game ended (bool)
        """
        # Preprocess frames: numpy (210×160×3) → tensor (1×3×128×128)
        state      = preprocess_frame(state)
        next_state = preprocess_frame(next_state)

        # Store this experience as a tuple in memory
        self.memory.append((state, action, reward, next_state, done))

        # Only start learning once we have enough memories to fill a minibatch
        if len(self.memory) > MINIBATCH_SIZE:
            experiences = random.sample(self.memory, k=MINIBATCH_SIZE)
            self.learn(experiences, DISCOUNT_FACTOR)

    # ──────────────────────────────────────────────────────────────────────────
    def act(self, state, epsilon=0.0):
        """
        Choose an action using the epsilon-greedy policy.

        With probability (1 - epsilon): EXPLOIT — pick the best known action.
        With probability epsilon       : EXPLORE — pick a random action.

        Parameters:
          state   — current game frame
          epsilon — exploration rate (float between 0 and 1)

        Returns:
          action  — integer index of the chosen action
        """
        state = preprocess_frame(state).to(self.device)

        # Switch to eval mode (disables dropout/batchnorm randomness) for inference
        self.local_qnetwork.eval()
        with torch.no_grad():   # Don't track gradients — we're not learning here
            action_values = self.local_qnetwork(state)   # Shape: (1, num_actions)
        self.local_qnetwork.train()   # Switch back to training mode

        if random.random() > epsilon:
            # EXPLOIT: pick the action with the highest Q-value
            return int(np.argmax(action_values.cpu().data.numpy()))
        else:
            # EXPLORE: pick a completely random action
            return random.randint(0, self.action_size - 1)

    # ──────────────────────────────────────────────────────────────────────────
    def learn(self, experiences, discount_factor):
        """
        Update the local Q-network using a minibatch of past experiences.

        The learning target (TD target) is:
            Q_target = reward + γ * max_a Q_target_network(next_state, a)

        This is the Bellman equation — the idea that the value of being in a
        state equals the immediate reward PLUS the discounted best future value.

        We minimise: MSE( Q_local(state, action),  Q_target )
        """
        # ── Unpack the batch of experiences ────────────────────────────────────
        states, actions, rewards, next_states, dones = zip(*experiences)

        # Convert to PyTorch tensors on the correct device
        # states: list of (1,3,128,128) tensors → stack into (64,3,128,128)
        states      = torch.cat(states).float().to(self.device)

        # actions: list of integers → tensor of shape (64, 1)
        actions     = torch.tensor(actions, dtype=torch.long).unsqueeze(1).to(self.device)

        # rewards: list of floats → tensor of shape (64, 1)
        rewards     = torch.tensor(rewards, dtype=torch.float).unsqueeze(1).to(self.device)

        next_states = torch.cat(next_states).float().to(self.device)

        # dones: list of booleans → tensor of shape (64, 1)
        dones       = torch.tensor(dones, dtype=torch.bool).unsqueeze(1).to(self.device)

        # ── Compute TD targets using the TARGET network ────────────────────────
        # detach() prevents gradients from flowing into the target network
        # .max(1)[0] picks the maximum Q-value across all actions
        next_q_values = self.target_qnetwork(next_states).detach().max(1)[0].unsqueeze(1)

        # Bellman equation: if done, no future reward; if not done, add discounted future
        # ~dones inverts booleans: True (done) → False (don't add future), False → True
        q_targets = rewards + (discount_factor * next_q_values * (~dones))

        # ── Compute current Q-values from the LOCAL network ───────────────────
        # .gather(1, actions) picks the Q-value for the action that was actually taken
        q_expected = self.local_qnetwork(states).gather(1, actions)

        # ── Compute Mean Squared Error loss ───────────────────────────────────
        # MSE measures how far our predictions are from the targets
        loss = F.mse_loss(q_expected, q_targets)

        # ── Backpropagation: adjust the local network weights ─────────────────
        self.optimizer.zero_grad()   # Clear old gradients
        loss.backward()              # Compute new gradients
        self.optimizer.step()        # Update weights

        # ── Soft-update the target network ────────────────────────────────────
        self.soft_update(self.local_qnetwork, self.target_qnetwork, SOFT_UPDATE_TAU)

    # ──────────────────────────────────────────────────────────────────────────
    def soft_update(self, local_model, target_model, tau):
        """
        Gradually blend the local network's weights into the target network.

        Formula: θ_target = τ * θ_local + (1 - τ) * θ_target

        With τ = 0.001:
          - 99.9% of the target stays the same
          - 0.1% is nudged toward the local network
        This keeps the target stable while slowly tracking improvements.
        """
        for target_param, local_param in zip(target_model.parameters(),
                                             local_model.parameters()):
            target_param.data.copy_(
                tau * local_param.data + (1.0 - tau) * target_param.data
            )


# ── Initialise the agent ───────────────────────────────────────────────────────
agent = Agent(number_action)
print(f"\nAgent ready with {number_action} possible actions.")


---
## 🏋️ Section 8 — Train the Agent

**What we're doing here:**  
This is the main training loop — where the AI actually *learns* to play Ms. Pac-Man.

### How One Episode Works
1. The game resets — Pac-Man starts at the beginning of the maze
2. For each timestep:
   - The AI looks at the current screen
   - It picks an action (using epsilon-greedy)
   - The game advances one frame
   - We store what happened in memory
   - The AI learns from a random batch of past experiences
3. The episode ends when Pac-Man loses all lives (`done = True`) or the timestep limit is reached

### Epsilon Decay — From Explorer to Expert
| Phase | Epsilon | Behaviour |
|---|---|---|
| Start | 1.0 | 100% random — pure exploration |
| Mid | ~0.5 | Mix of exploration and exploitation |
| End | 0.01 | 99% using learned knowledge |

Every episode: `ε = max(ε_min, ε × decay_rate)`

### What "Solved" Means
We consider the environment "solved" when the AI averages **500+ points** over 100 consecutive episodes. A well-trained agent can reach much higher scores.

> ⏱️ **Training time:** On a free Colab GPU (T4), training 2000 episodes takes roughly **1–2 hours**. The AI won't play well until it has trained for at least a few hundred episodes.


In [ ]:
# ── Training hyperparameters ──────────────────────────────────────────────────
NUMBER_EPISODES           = 2000   # Total episodes to train for
MAX_TIMESTEPS_PER_EPISODE = 1000   # Max steps before an episode is forced to end
EPSILON_START             = 1.0    # Start fully random
EPSILON_END               = 0.01   # Never go below 1% random
EPSILON_DECAY             = 0.995  # Multiply epsilon by this after each episode
SCORE_THRESHOLD           = 500.0  # If avg score over 100 episodes reaches this, stop

# ── Initialise training state ─────────────────────────────────────────────────
epsilon       = EPSILON_START
scores_window = deque(maxlen=100)   # Rolling window — only keep last 100 scores
best_avg      = -float('inf')       # Track best average score seen so far

print("Starting training...")
print(f"  Episodes         : {NUMBER_EPISODES}")
print(f"  Max steps/episode: {MAX_TIMESTEPS_PER_EPISODE}")
print(f"  Epsilon start    : {EPSILON_START} → {EPSILON_END} (decay {EPSILON_DECAY})")
print(f"  Solve threshold  : avg score ≥ {SCORE_THRESHOLD} over 100 episodes\n")

# ── Main training loop ────────────────────────────────────────────────────────
for episode in range(1, NUMBER_EPISODES + 1):

    # Reset the game to the beginning; get the initial frame
    state, _ = env.reset()
    score = 0   # Accumulate points earned this episode

    # ── Inner loop: play one episode step by step ─────────────────────────────
    for t in range(MAX_TIMESTEPS_PER_EPISODE):

        # 1. Choose an action (greedy with probability 1-ε, random with ε)
        action = agent.act(state, epsilon)

        # 2. Execute the action in the game
        #    Returns: next frame, reward earned, whether game ended, info dicts
        next_state, reward, done, _, _ = env.step(action)

        # 3. Store experience in memory and (if enough memories) learn from a batch
        agent.step(state, action, reward, next_state, done)

        # 4. Move to the next state
        state  = next_state
        score += reward

        # 5. If the game ended (Pac-Man lost all lives), stop this episode
        if done:
            break

    # ── After each episode ────────────────────────────────────────────────────

    # Store the episode score
    scores_window.append(score)

    # Decay epsilon: reduce exploration as the agent gets better
    epsilon = max(EPSILON_END, EPSILON_DECAY * epsilon)

    avg_score = np.mean(scores_window)

    # Track best model and save checkpoint whenever average improves
    if avg_score > best_avg:
        best_avg = avg_score
        torch.save(agent.local_qnetwork.state_dict(), 'checkpoint_best.pth')

    # Print progress on the same line (overwrite) for clean output
    print(f'\rEpisode {episode:4d}  |  Score: {score:6.1f}  |  Avg(100): {avg_score:6.2f}'
          f'  |  ε: {epsilon:.4f}', end="")

    # Every 100 episodes: print a full newline summary
    if episode % 100 == 0:
        print(f'\rEpisode {episode:4d}  |  Score: {score:6.1f}  |  Avg(100): {avg_score:6.2f}'
              f'  |  ε: {epsilon:.4f}')

    # Early stopping if solved
    if avg_score >= SCORE_THRESHOLD:
        print(f'\n\n✅ Environment SOLVED in {episode - 100} episodes!')
        print(f'   Average Score: {avg_score:.2f}')
        torch.save(agent.local_qnetwork.state_dict(), 'checkpoint_solved.pth')
        break

print(f'\n\n🏁 Training finished. Best average score: {best_avg:.2f}')


---
## 🎬 Section 9 — Visualise the Trained Agent Playing

**What we're doing here:**  
We let the trained AI play a full game of Ms. Pac-Man and record it as a video, then display it right here in the notebook!

**Key difference from training:**
- We pass `epsilon=0.0` to `agent.act()` — no random actions. The AI always picks what it thinks is the best move.
- We set `render_mode='rgb_array'` so the environment returns each frame as an image we can save.

> 🎥 The video will appear inline in the cell below after the agent finishes one full game.


In [ ]:
import glob
import io
import base64
import imageio
from IPython.display import HTML, display

def show_video_of_model(agent, env_name):
    """
    Run the trained agent through one full episode with no exploration,
    record all rendered frames, and save as 'video.mp4'.

    Parameters:
      agent    — the trained Agent object
      env_name — Gymnasium environment ID string
    """
    # Create a fresh environment with video rendering enabled
    env_vis = gym.make(env_name, render_mode='rgb_array')
    state, _ = env_vis.reset()

    done   = False
    frames = []
    total_score = 0

    while not done:
        # Capture current frame (RGB array) and store it
        frame = env_vis.render()
        frames.append(frame)

        # Act greedily — epsilon=0 means NO random actions, purely learned behaviour
        action = agent.act(state, epsilon=0.0)

        state, reward, done, _, _ = env_vis.step(action)
        total_score += reward

    env_vis.close()

    # Save all frames as an MP4 video at 30 frames per second
    imageio.mimsave('video.mp4', frames, fps=30)
    print(f"✅ Video saved! ({len(frames)} frames, score: {total_score:.0f} points)")


def show_video():
    """Display the most recently saved .mp4 video inline in the notebook."""
    mp4_files = glob.glob('*.mp4')
    if mp4_files:
        mp4     = mp4_files[0]
        video   = io.open(mp4, 'r+b').read()
        encoded = base64.b64encode(video)
        display(HTML(data='''
            <video alt="Ms. Pac-Man AI Agent" autoplay loop controls style="height:400px;">
                <source src="data:video/mp4;base64,{0}" type="video/mp4" />
            </video>
        '''.format(encoded.decode('ascii'))))
    else:
        print("❌ No video file found. Run show_video_of_model() first.")


# ── Record and display the trained agent ──────────────────────────────────────
print("Recording the agent playing Ms. Pac-Man...")
show_video_of_model(agent, 'ALE/MsPacman-v5')
show_video()


---
## 💾 Section 10 — Save and Load the Model

**Saving** lets you download your trained AI and use it later without retraining from scratch.

**Loading** lets you continue training from a checkpoint or use a pre-trained model directly.


In [ ]:
# ── Save the trained model weights ───────────────────────────────────────────
torch.save(agent.local_qnetwork.state_dict(), 'pacman_dcqn_final.pth')
print("✅ Model saved as 'pacman_dcqn_final.pth'")

# ── Download the file (Colab only) ────────────────────────────────────────────
try:
    from google.colab import files
    files.download('pacman_dcqn_final.pth')
    print("📥 Download started!")
except ImportError:
    print("(Not running in Colab — file saved locally as 'pacman_dcqn_final.pth')")

# ───────────────────────────────────────────────────────────────────────────────
# To LOAD a saved model later, run:
#
# agent_loaded = Agent(number_action)
# agent_loaded.local_qnetwork.load_state_dict(torch.load('pacman_dcqn_final.pth'))
# agent_loaded.local_qnetwork.eval()   # Set to evaluation mode
# print("Model loaded successfully!")
# ───────────────────────────────────────────────────────────────────────────────


---
## 🎉 Congratulations! You've trained an AI to play Ms. Pac-Man!

### 🔁 Quick Recap — What You Built

| Component | Role |
|---|---|
| `Network` (CNN) | The AI's "brain" — sees game pixels and outputs Q-values |
| `preprocess_frame` | The AI's "eyes" — converts raw pixels into a clean tensor |
| `Agent.act()` | The AI's "decision maker" — epsilon-greedy action selection |
| `Agent.step()` | The AI's "short-term memory" — stores and triggers learning |
| `Agent.learn()` | The AI's "learning engine" — Bellman equation + backprop |
| `Agent.soft_update()` | Keeps the target network stable while the local network improves |

---

### 💡 Ideas to Try Next

1. **Increase `MEMORY_SIZE`** to 100,000 for richer experience replay
2. **Try Double DQN** — use the local network to *select* the best action, but the target network to *evaluate* it (reduces overestimation bias)
3. **Try Dueling DQN** — split the output into a *Value* stream and an *Advantage* stream
4. **Train on a different Atari game** — just change `"ALE/MsPacman-v5"` to any other ALE environment
5. **Add Prioritised Experience Replay** — sample experiences that the AI found surprising more often

---
> 🔬 *This notebook implements the classic DQN algorithm introduced by DeepMind in their 2015 Nature paper "Human-level control through deep reinforcement learning".*
